In [1]:
import pandas as pd
import numpy as np
import re
import ast

In [2]:
df = pd.read_parquet("./data/train-00000-of-00001.parquet")
df.head()

,id,local_id,specialty,author,reviewer,pool,suggested_scenario,documents,structured_abstract
0,ANATOMOPATHOLOGIE-00001,00001,ANATOMOPATHOLOGIE,ADU,CJA,General,"{'name': 'Chantale Peruchon', 'age': {'value':...","{'type': ['ANAPATH'], 'header': ['COMPTE-RENDU...","{'primary_diagnosis': None, 'primary_procedure..."
1,ANATOMOPATHOLOGIE-00002,00002,ANATOMOPATHOLOGIE,ADU,CJA,General,"{'name': 'Colette Lorilleux', 'age': {'value':...","{'type': ['ANAPATH'], 'header': ['COMPTE-RENDU...","{'primary_diagnosis': None, 'primary_procedure..."
2,ANATOMOPATHOLOGIE-00003,00003,ANATOMOPATHOLOGIE,ADU,CJA,General,"{'name': 'Kaena Koska', 'age': {'value': 69, '...","{'type': ['ANAPATH'], 'header': ['COMPTE-RENDU...","{'primary_diagnosis': None, 'primary_procedure..."
3,ANATOMOPATHOLOGIE-00004,00004,ANATOMOPATHOLOGIE,ADU,CJA,General,"{'name': 'Annette Tournoux', 'age': {'value': ...","{'type': ['ANAPATH'], 'header': ['COMPTE-RENDU...","{'primary_diagnosis': None, 'primary_procedure..."
4,ANATOMOPATHOLOGIE-00005,00005,ANATOMOPATHOLOGIE,ADU,CJA,General,"{'name': 'Elisabeta Riaud', 'age': {'value': 7...","{'type': ['ANAPATH'], 'header': ['COMPTE-RENDU...","{'primary_diagnosis': None, 'primary_procedure..."


In [3]:
df.columns

Index(['id', 'local_id', 'specialty', 'author', 'reviewer', 'pool',
       'suggested_scenario', 'documents', 'structured_abstract'],
      dtype='str')

In [4]:
df.dtypes

id                        str
local_id                  str
specialty                 str
author                    str
reviewer                  str
pool                      str
suggested_scenario     object
documents              object
structured_abstract    object
dtype: object

In [5]:
print(df.loc[0])

id                                               ANATOMOPATHOLOGIE-00001
local_id                                                           00001
specialty                                              ANATOMOPATHOLOGIE
author                                                               ADU
reviewer                                                             CJA
pool                                                             General
suggested_scenario     {'name': 'Chantale Peruchon', 'age': {'value':...
documents              {'type': ['ANAPATH'], 'header': ['COMPTE-RENDU...
structured_abstract    {'primary_diagnosis': None, 'primary_procedure...
Name: 0, dtype: object


In [6]:
print(df.loc[0, 'documents'])

{'type': array(['ANAPATH'], dtype=object), 'header': array(['COMPTE-RENDU ANATOMO PATHOLOGIQUE'], dtype=object), 'text': array(["COMPTE-RENDU ANATOMO PATHOLOGIQUE\n\nPatient : Chantale Peruchon\n\nNé le : 05/07/1949\n\nSexe : F\n\nPrélevé le Reçu le : 19/05/2025\n\nPrescripteur : Dr Clélya Mouraille\n\nRenseignements cliniques :\n\nImmunodéprimé. Contexte de leucémie lymphoïde chronique allogreffée. Pneumopathie du lobe supérieur gauche.\n\nBIOPSIES BRONCHIQUES\n\nMACROSCOPIE\n\nPot 1 / Bronche du culmen : trois fragments mesurant de 1 à 2 mm.\n\nPot 2 / Bronche du culmen : trois fragments mesurant de 1 à 2 mm.\n\nPot 3 / Eperon LSG / LIG : trois fragments mesurant de 1 à 2 mm.\n\nPot 4 / Carène : trois fragments mesurant 1 à 2 mm.\n\nInclusion en totalité en 4 blocs. Analyse après coloration HES sur 3 niveaux de coupe/bloc.\n\nMICROSCOPIE\n\nLes prélèvements des pots n° 1 et 2 ont un aspect comparable. La muqueuse bronchique est bordée d'un épithélium de type respiratoire, sans atypie

In [7]:
print(f'Taille du dataset : {df.shape}\nIds differents : {len(df['id'].unique())}')

Taille du dataset : (4254, 9)
Ids differents : 4254


In [8]:
df['specialty'].unique()

<ArrowStringArray>
[         'ANATOMOPATHOLOGIE',             'CANCERO ADULTE',
                'CARDIOLOGIE',     'CHIR ORTHO ET TRAUMATO',
          'CHIR.CARDIO-VASC.',        'CHIRURGIE VISCERALE',
                'GYNECOLOGIE',       'HEMATOLOGIE CLINIQUE',
       'HEPATO-GASTRO-ENTERO',      'MALADIES INFECTIEUSES',
       'MEDECINE GERIATRIQUE', 'MEDECINE INTER-SPECIALITES',
           'MEDECINE INTERNE',       'MEDECINE PEDIATRIQUE',
                'NEPHROLOGIE',                 'NEUROLOGIE',
                'OBSTETRIQUE',                'PNEUMOLOGIE',
                'REANIMATION',                   'UROLOGIE']
Length: 20, dtype: str

In [9]:
df['specialty'].value_counts()

specialty
PNEUMOLOGIE                   531
CANCERO ADULTE                507
ANATOMOPATHOLOGIE             432
CHIRURGIE VISCERALE           295
MALADIES INFECTIEUSES         285
NEUROLOGIE                    251
CHIR ORTHO ET TRAUMATO        241
MEDECINE INTERNE              228
GYNECOLOGIE                   212
OBSTETRIQUE                   174
UROLOGIE                      173
HEPATO-GASTRO-ENTERO          170
MEDECINE GERIATRIQUE          155
HEMATOLOGIE CLINIQUE          153
NEPHROLOGIE                   104
CARDIOLOGIE                    96
MEDECINE INTER-SPECIALITES     82
MEDECINE PEDIATRIQUE           82
REANIMATION                    44
CHIR.CARDIO-VASC.              39
Name: count, dtype: int64

In [10]:
df['pool'].unique()

<ArrowStringArray>
[                                 'General',
            'CU 5a - Oncology (biomarkers)',
 'CU 5b - Oncology (response to treatment)',
                     'CU 2 - ICD-10 coding',
                      'CU 6 - Infectiology']
Length: 5, dtype: str

In [11]:
df['pool'].value_counts()

pool
General                                     3577
CU 2 - ICD-10 coding                         288
CU 5a - Oncology (biomarkers)                147
CU 6 - Infectiology                          134
CU 5b - Oncology (response to treatment)     108
Name: count, dtype: int64

In [12]:
df_documents = df['documents'].apply(pd.Series)

In [13]:
df_documents.head()

,type,header,text,word_count
0,[ANAPATH],[COMPTE-RENDU ANATOMO PATHOLOGIQUE],[COMPTE-RENDU ANATOMO PATHOLOGIQUE\n\nPatient ...,[347]
1,[ANAPATH],[COMPTE-RENDU ANATOMO PATHOLOGIQUE],[COMPTE-RENDU ANATOMO PATHOLOGIQUE\n\nPatient ...,[320]
2,[ANAPATH],[COMPTE-RENDU ANATOMO PATHOLOGIQUE],[COMPTE-RENDU ANATOMO PATHOLOGIQUE\n\nPatient ...,[508]
3,[ANAPATH],[COMPTE-RENDU ANATOMO PATHOLOGIQUE],[COMPTE-RENDU ANATOMO PATHOLOGIQUE\n\nPatient ...,[739]
4,[ANAPATH],[COMPTE-RENDU ANATOMO PATHOLOGIQUE],[COMPTE-RENDU ANATOMO PATHOLOGIQUE\n\nPatient ...,[717]


In [14]:
df_documents.dtypes

type          object
header        object
text          object
word_count    object
dtype: object

In [15]:
df_documents['text_str'] = df_documents['text'].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else str(x))

In [16]:
df_documents.head()

,type,header,text,word_count,text_str
0,[ANAPATH],[COMPTE-RENDU ANATOMO PATHOLOGIQUE],[COMPTE-RENDU ANATOMO PATHOLOGIQUE\n\nPatient ...,[347],"[""COMPTE-RENDU ANATOMO PATHOLOGIQUE\n\nPatient..."
1,[ANAPATH],[COMPTE-RENDU ANATOMO PATHOLOGIQUE],[COMPTE-RENDU ANATOMO PATHOLOGIQUE\n\nPatient ...,[320],"[""COMPTE-RENDU ANATOMO PATHOLOGIQUE\n\nPatient..."
2,[ANAPATH],[COMPTE-RENDU ANATOMO PATHOLOGIQUE],[COMPTE-RENDU ANATOMO PATHOLOGIQUE\n\nPatient ...,[508],"[""COMPTE-RENDU ANATOMO PATHOLOGIQUE\n\nPatient..."
3,[ANAPATH],[COMPTE-RENDU ANATOMO PATHOLOGIQUE],[COMPTE-RENDU ANATOMO PATHOLOGIQUE\n\nPatient ...,[739],"[""COMPTE-RENDU ANATOMO PATHOLOGIQUE\n\nPatient..."
4,[ANAPATH],[COMPTE-RENDU ANATOMO PATHOLOGIQUE],[COMPTE-RENDU ANATOMO PATHOLOGIQUE\n\nPatient ...,[717],"[""COMPTE-RENDU ANATOMO PATHOLOGIQUE\n\nPatient..."


In [17]:
type(df_documents.loc[0,'word_count'])

numpy.ndarray

In [18]:
len(df_documents.loc[0,'word_count'])

1

In [19]:
type(df_documents.loc[0,'word_count'][0])

numpy.int32

In [20]:
df_documents['word_count'] = df_documents['word_count'].tolist()

In [21]:
(type(df_documents['word_count']))

pandas.Series

In [22]:
def extraire_entier(x):
    return int(np.sum(x))

df_documents['word_count_int'] = df_documents['word_count'].apply(extraire_entier)

In [23]:
df_documents.dtypes

type              object
header            object
text              object
word_count        object
text_str             str
word_count_int     int64
dtype: object

In [24]:
df_documents.head()

,type,header,text,word_count,text_str,word_count_int
0,[ANAPATH],[COMPTE-RENDU ANATOMO PATHOLOGIQUE],[COMPTE-RENDU ANATOMO PATHOLOGIQUE\n\nPatient ...,[347],"[""COMPTE-RENDU ANATOMO PATHOLOGIQUE\n\nPatient...",347
1,[ANAPATH],[COMPTE-RENDU ANATOMO PATHOLOGIQUE],[COMPTE-RENDU ANATOMO PATHOLOGIQUE\n\nPatient ...,[320],"[""COMPTE-RENDU ANATOMO PATHOLOGIQUE\n\nPatient...",320
2,[ANAPATH],[COMPTE-RENDU ANATOMO PATHOLOGIQUE],[COMPTE-RENDU ANATOMO PATHOLOGIQUE\n\nPatient ...,[508],"[""COMPTE-RENDU ANATOMO PATHOLOGIQUE\n\nPatient...",508
3,[ANAPATH],[COMPTE-RENDU ANATOMO PATHOLOGIQUE],[COMPTE-RENDU ANATOMO PATHOLOGIQUE\n\nPatient ...,[739],"[""COMPTE-RENDU ANATOMO PATHOLOGIQUE\n\nPatient...",739
4,[ANAPATH],[COMPTE-RENDU ANATOMO PATHOLOGIQUE],[COMPTE-RENDU ANATOMO PATHOLOGIQUE\n\nPatient ...,[717],"[""COMPTE-RENDU ANATOMO PATHOLOGIQUE\n\nPatient...",717


In [25]:
df = pd.concat([df, df_documents], axis=1)

In [26]:
df = df.drop(columns=['documents', 'text', 'word_count'])
df = df.rename(columns={
    "word_count_int": 'word_count',
    "text_str": 'text'
})

In [27]:
df.dtypes

id                        str
local_id                  str
specialty                 str
author                    str
reviewer                  str
pool                      str
suggested_scenario     object
structured_abstract    object
type                   object
header                 object
text                      str
word_count              int64
dtype: object

In [28]:
df.loc[0, 'type']

array(['ANAPATH'], dtype=object)

In [29]:
df.loc[0, 'header']

array(['COMPTE-RENDU ANATOMO PATHOLOGIQUE'], dtype=object)

In [30]:
df.loc[0, 'type'][0]

'ANAPATH'

In [31]:
df.loc[0, 'header'][0]

'COMPTE-RENDU ANATOMO PATHOLOGIQUE'

In [32]:
type(df.loc[0, 'type'][0])

str

In [33]:
def extraire_str(x):
    return str(x[0])
df['type'] = df['type'].apply(extraire_str)
df['header'] = df['header'].apply(extraire_str)

In [34]:
df.dtypes

id                        str
local_id                  str
specialty                 str
author                    str
reviewer                  str
pool                      str
suggested_scenario     object
structured_abstract    object
type                      str
header                    str
text                      str
word_count              int64
dtype: object

In [35]:
df.head()

,id,local_id,specialty,author,reviewer,pool,suggested_scenario,structured_abstract,type,header,text,word_count
0,ANATOMOPATHOLOGIE-00001,00001,ANATOMOPATHOLOGIE,ADU,CJA,General,"{'name': 'Chantale Peruchon', 'age': {'value':...","{'primary_diagnosis': None, 'primary_procedure...",ANAPATH,COMPTE-RENDU ANATOMO PATHOLOGIQUE,"[""COMPTE-RENDU ANATOMO PATHOLOGIQUE\n\nPatient...",347
1,ANATOMOPATHOLOGIE-00002,00002,ANATOMOPATHOLOGIE,ADU,CJA,General,"{'name': 'Colette Lorilleux', 'age': {'value':...","{'primary_diagnosis': None, 'primary_procedure...",ANAPATH,COMPTE-RENDU ANATOMO PATHOLOGIQUE,"[""COMPTE-RENDU ANATOMO PATHOLOGIQUE\n\nPatient...",320
2,ANATOMOPATHOLOGIE-00003,00003,ANATOMOPATHOLOGIE,ADU,CJA,General,"{'name': 'Kaena Koska', 'age': {'value': 69, '...","{'primary_diagnosis': None, 'primary_procedure...",ANAPATH,COMPTE-RENDU ANATOMO PATHOLOGIQUE,"[""COMPTE-RENDU ANATOMO PATHOLOGIQUE\n\nPatient...",508
3,ANATOMOPATHOLOGIE-00004,00004,ANATOMOPATHOLOGIE,ADU,CJA,General,"{'name': 'Annette Tournoux', 'age': {'value': ...","{'primary_diagnosis': None, 'primary_procedure...",ANAPATH,COMPTE-RENDU ANATOMO PATHOLOGIQUE,"[""COMPTE-RENDU ANATOMO PATHOLOGIQUE\n\nPatient...",739
4,ANATOMOPATHOLOGIE-00005,00005,ANATOMOPATHOLOGIE,ADU,CJA,General,"{'name': 'Elisabeta Riaud', 'age': {'value': 7...","{'primary_diagnosis': None, 'primary_procedure...",ANAPATH,COMPTE-RENDU ANATOMO PATHOLOGIQUE,"[""COMPTE-RENDU ANATOMO PATHOLOGIQUE\n\nPatient...",717


In [36]:
df['word_count'].describe()

count    4254.000000
mean      929.160085
std       571.981903
min       167.000000
25%       572.000000
50%       799.000000
75%      1127.000000
max      7006.000000
Name: word_count, dtype: float64

In [37]:
df_documents["word_count"]

0                [347]
1                [320]
2                [508]
3                [739]
4                [717]
             ...      
4249    [191, 137, 49]
4250    [246, 144, 49]
4251    [308, 219, 49]
4252    [296, 328, 94]
4253    [189, 332, 80]
Name: word_count, Length: 4254, dtype: object

In [38]:
print(df_documents.loc[4249])

type                                                [CRC, CRO, CRH]
header            [Compte rendu de consultation de Chrurgie urol...
text              [Compte rendu de consultation de Chrurgie urol...
word_count                                           [191, 137, 49]
text_str          ["Compte rendu de consultation de Chrurgie uro...
word_count_int                                                  377
Name: 4249, dtype: object


In [39]:
print(df_documents.loc[4249, 'text'])

["Compte rendu de consultation de Chrurgie urologique\n\nLe 21.06.2025\n\nJe vois ce jour en consultation Monsieur Marcel Vermeglio, pour tuméfaction scrotale droite.\n\nATCD : angor d'effort, HTA, diabète de type 2\n\nTTT : RAMIPRIL, METFORMINE\n\nAllergies : 0\n\nIl s'agit d'un patient de 78 ans présentant une tuméfaction scrotale droite importante évoluant depuis 6 mois.\n\nIl a réalisé une échographie en ville qui retrouve une hydrocèle droite mesurant 12X9 mm, sans anomalie du testicule associée.\n\nLe patient se dit gêné par cette tuméfaction et souhaite une intervention.\n\nJe lui explique qu'il est possible de réaliser une cure d'hydrocèle avec résection de la vaginale pour éviter les récidives.\n\nJe lui expose les modalités et risques de l'intervention (hématome, surinfection).\n\nLe patient consent à l'intervention.\n\nJe lui remet la fiche AFU et je programme la chirurgie le 15.07.2025.\n\nDr Josiane Sourbes.\n"
 "Compte rendu opératoire de Chrurgie urologique\n\nRappel cli

In [40]:
print(df_documents.loc[4249, 'header'])

['Compte rendu de consultation de Chrurgie urologique'
 'Compte rendu opératoire de Chrurgie urologique'
 "Compte rendu d'hospitalisation du service de Chrurgie urologique"]


In [41]:
type(df_documents.loc[4249, 'header'])

numpy.ndarray

In [42]:
def compter_elements(x):
    # Retourne la taille si c'est un tableau NumPy ou une liste
    if isinstance(x, (np.ndarray, list)):
        return len(x)
    # Retourne 0 si la cellule est vide
    elif pd.isna(x):
        return 0
    # Retourne 1 s'il s'agit d'une chaîne de caractères simple
    else:
        return 1

df_documents['nombre_docs'] = len(df_documents['header'])

total_documents = df_documents['nombre_docs'].sum()

print(f"Nombre total de documents individuels : {total_documents}")

Nombre total de documents individuels : 18096516


In [43]:
print(len(df_documents.loc[4249, 'header']))

3


In [45]:
df.loc[0, "suggested_scenario"]

{'name': 'Chantale Peruchon',
 'age': {'value': 76, 'unit': 'ans'},
 'sex': 'F',
 'admission_mode': 'domicile',
 'discharge_mode': 'retour à domicile',
 'primary_procedure': None,
 'primary_diagnosis': {'code': array(['C341'], dtype=object),
  'description': array(['Tumeur maligne du lobe supérieur, bronches ou poumon'],
        dtype=object)},
 'type_of_care': None}

In [48]:
df_suggested_scenario = df['suggested_scenario'].apply(pd.Series)

In [52]:
df = pd.concat(
    [df.drop(columns=["suggested_scenario"]), pd.json_normalize(df["suggested_scenario"])],
    axis=1
)

In [53]:
df.head()

,id,local_id,specialty,author,reviewer,pool,structured_abstract,type,header,text,...,name,sex,admission_mode,discharge_mode,primary_procedure,type_of_care,age.value,age.unit,primary_diagnosis.code,primary_diagnosis.description
0,ANATOMOPATHOLOGIE-00001,00001,ANATOMOPATHOLOGIE,ADU,CJA,General,"{'primary_diagnosis': None, 'primary_procedure...",ANAPATH,COMPTE-RENDU ANATOMO PATHOLOGIQUE,"[""COMPTE-RENDU ANATOMO PATHOLOGIQUE\n\nPatient...",...,Chantale Peruchon,F,domicile,retour à domicile,None,NaN,76,ans,[C341],"[Tumeur maligne du lobe supérieur, bronches ou..."
1,ANATOMOPATHOLOGIE-00002,00002,ANATOMOPATHOLOGIE,ADU,CJA,General,"{'primary_diagnosis': None, 'primary_procedure...",ANAPATH,COMPTE-RENDU ANATOMO PATHOLOGIQUE,"[""COMPTE-RENDU ANATOMO PATHOLOGIQUE\n\nPatient...",...,Colette Lorilleux,F,domicile,retour à domicile,None,NaN,59,ans,[C821],"[Lymphome folliculaire, grade II]"
2,ANATOMOPATHOLOGIE-00003,00003,ANATOMOPATHOLOGIE,ADU,CJA,General,"{'primary_diagnosis': None, 'primary_procedure...",ANAPATH,COMPTE-RENDU ANATOMO PATHOLOGIQUE,"[""COMPTE-RENDU ANATOMO PATHOLOGIQUE\n\nPatient...",...,Kaena Koska,F,domicile,retour à domicile,None,NaN,69,ans,[C508],[Tumeur maligne à localisations contiguës du s...
3,ANATOMOPATHOLOGIE-00004,00004,ANATOMOPATHOLOGIE,ADU,CJA,General,"{'primary_diagnosis': None, 'primary_procedure...",ANAPATH,COMPTE-RENDU ANATOMO PATHOLOGIQUE,"[""COMPTE-RENDU ANATOMO PATHOLOGIQUE\n\nPatient...",...,Annette Tournoux,F,domicile,retour à domicile,None,NaN,56,ans,[C504],[Tumeur maligne du quadrant supéro-externe du ...
4,ANATOMOPATHOLOGIE-00005,00005,ANATOMOPATHOLOGIE,ADU,CJA,General,"{'primary_diagnosis': None, 'primary_procedure...",ANAPATH,COMPTE-RENDU ANATOMO PATHOLOGIQUE,"[""COMPTE-RENDU ANATOMO PATHOLOGIQUE\n\nPatient...",...,Elisabeta Riaud,F,entrée par les urgences,retour à domicile,None,NaN,74,ans,[C509],"[Tumeur maligne du sein, sans précision]"


In [47]:
df.loc[4253, 'structured_abstract']

{'primary_diagnosis': {'code': array(['N201'], dtype=object),
  'description': array(["Calcul de l'uretère"], dtype=object)},
 'primary_procedure': {'code': array(['JCGE006'], dtype=object),
  'description': array(["Ablation et/ou fragmentation de calcul de l'uretère lombal, par urétéroscopie rétrograde"],
        dtype=object)},
 'admission_mode': None,
 'discharge_mode': None,
 'length_of_stay': {'value': 0.0, 'unit': 'jours'}}

In [54]:
df.loc[4253]

id                                                                  UROLOGIE-00308
local_id                                                                     00308
specialty                                                                 UROLOGIE
author                                                                         CLE
reviewer                                                                       ALE
pool                                                          CU 2 - ICD-10 coding
structured_abstract              {'primary_diagnosis': {'code': ['N201'], 'desc...
type                                                                           CRC
header                           Compte rendu de consultation de Chrurgie urolo...
text                             ["Compte rendu de consultation de Chrurgie uro...
word_count                                                                     601
name                                                                Haitem Nougier
sex 